In [0]:
import os

CATALOG_NAME = 'workspace'
SCHEMA_NAME = 'lab_2026'
VOLUME_DIR = os.path.abspath('/Volumes/{}/{}/'.format(CATALOG_NAME, SCHEMA_NAME))
RAW_DIR = os.path.join(VOLUME_DIR, 'raw')
COMPRESSED_RAW_DIR = os.path.join(RAW_DIR, '.compressed')
if not os.path.exists(COMPRESSED_RAW_DIR):
    raise Exception(f"{COMPRESSED_RAW_DIR} Not Found!")

INPUT_FILE = 'online_retail_II.csv'
compressed_file_path = os.path.join(COMPRESSED_RAW_DIR, INPUT_FILE + '.zip')
dbutils.fs.ls(compressed_file_path)

In [0]:
import zipfile

csv_file_path = os.path.join(RAW_DIR)
with zipfile.ZipFile(compressed_file_path, "r") as zip_ref:
    zip_ref.extractall(csv_file_path)
dbutils.fs.ls(csv_file_path)

In [0]:
spark.read.csv(RAW_DIR, header=True, inferSchema=False).limit(5).display()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType
bronze_schema = StructType([
    StructField("Invoice", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", StringType(), True),
    StructField("InvoiceDate", StringType(), True),
    StructField("Price", StringType(), True),
    StructField("Customer ID", StringType(), True),
    StructField("Country", StringType(), True)
])
checkpoint_location = os.path.join(RAW_DIR, '.checkpoints', 'stream_online_retail')

In [0]:
dbutils.fs.ls('dbfs:/Volumes/workspace/lab_2026/raw/.checkpoints/')
# dbutils.fs.rm('dbfs:/Volumes/workspace/lab_2026/raw/.checkpoints/', True)

In [0]:
source_df = (
    spark.readStream
    .format("csv")
    .schema(bronze_schema)
    .option("header", True)
    .option("maxFilesPerTrigger", 1)
    .option("checkpointLocation", checkpoint_location)
    .load(RAW_DIR)
)

import re
renamed_columns = dict()
for col in source_df.columns:
    renamed_columns[col] = re.sub(r'[^a-zA-Z0-9]', '_', col.lower())

from pyspark.sql import functions as F
raw_df = (
    source_df
    .withColumn('_hash_md5', F.md5(F.concat_ws(',', *source_df.columns)))
    .withColumn('_ingest_timestamp', F.current_timestamp())
    .withColumn('_ingest_author', F.current_user())
    .withColumn('_source_file', F.col("_metadata.file_path"))
    .withColumnsRenamed(renamed_columns)
)

In [0]:
query = (
    raw_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_location)
    .trigger(availableNow=True)
    .table("lab_2026.stream_online_retail")
)

In [0]:
%sql

select * from lab_2026.stream_online_retail
order by `_ingest_timestamp` desc
limit 20